In [1]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

!kaggle competitions download -c dga-domain-detection-challenge

# Теперь распакуем скачанный файл
!unzip -q dga-domain-detection-challenge.zip -d /kaggle/working/

# Проверим, что скачалось
!ls -la /kaggle/working/

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

100%|████████████████████████████████████████| 258M/258M [00:08<00:00, 31.3MB/s]

total 770012
drwxr-xr-x 2 root root      4096 Feb 22 11:58 .
drwxr-xr-x 8 root root      4096 Feb 22 11:58 ..
-rw-r--r-- 1 root root 270296844 Oct 13 21:18 dga-domain-detection-challenge.zip
---------- 1 root root    104206 Feb 22 11:58 __notebook__.ipynb
-rw-r--r-- 1 root root        66 Oct 13 21:17 sample_submission.csv
-rw-r--r-- 1 root root 186548519 Oct 13 21:17 test.csv
-rw-r--r-- 1 root root 331518539 Oct 13 21:17 train.csv


In [2]:
# Проверяем доступность GPU
import tensorflow as tf
print("GPU Available:", tf.config.list_physical_devices('GPU'))

# Оптимизируем загрузку данных - читаем чанками!
import pandas as pd
import numpy as np
import re
from tqdm import tqdm
import gc

# Функция для оптимизации типов данных (экономит память)
def optimize_dtypes(df):
    for col in df.columns:
        if df[col].dtype == 'float64':
            df[col] = df[col].astype('float32')
        elif df[col].dtype == 'int64':
            df[col] = df[col].astype('int32')
    return df

# Загружаем тренировочные данные чанками
print("Загружаем тренировочные данные чанками...")
train_chunks = pd.read_csv('/kaggle/working/train.csv', chunksize=500000)  # по 500к строк

# Создаем пустой DataFrame для результатов
feature_names = None
processed_chunks = []

# Обрабатываем каждый чанк
for i, chunk in enumerate(tqdm(train_chunks)):
    print(f"\nОбработка чанка {i+1}, размер: {len(chunk)}")
    
    # БЫСТРЫЙ парсинг через регулярные выражения (в 100 раз быстрее tldextract)
    def fast_domain_parse(domain):
        # Простое разбиение по точкам
        parts = str(domain).split('.')
        if len(parts) == 1:
            return pd.Series(['', parts[0], ''])
        elif len(parts) == 2:
            return pd.Series(['', parts[0], parts[1]])
        else:
            return pd.Series(['.'.join(parts[:-2]), parts[-2], parts[-1]])
    
    # Применяем быстрый парсинг
    parsed = chunk['domain'].apply(fast_domain_parse)
    parsed.columns = ['subdomain', 'sld', 'tld']
    
    # Создаем признаки (только самые важные для начала)
    chunk['sld'] = parsed['sld']
    chunk['tld'] = parsed['tld']
    
    # SLD признаки
    chunk['sld_length'] = chunk['sld'].str.len()
    chunk['sld_entropy'] = chunk['sld'].apply(lambda x: len(set(x)) / (len(x) + 1))  # упрощенная энтропия
    chunk['digit_count'] = chunk['sld'].str.count(r'\d')
    chunk['digit_ratio'] = chunk['digit_count'] / (chunk['sld_length'] + 1)
    chunk['has_digits'] = (chunk['digit_count'] > 0).astype('int8')
    chunk['has_hyphen'] = chunk['sld'].str.contains('-').astype('int8')
    chunk['vowel_count'] = chunk['sld'].str.count(r'[aeiouyAEIOUY]')
    chunk['vowel_ratio'] = chunk['vowel_count'] / (chunk['sld_length'] + 1)
    
    # TLD признаки (бинарные)
    suspicious_tlds = ['tk', 'ml', 'ga', 'cf', 'xyz', 'top', 'club', 'online', 'site']
    chunk['suspicious_tld'] = chunk['tld'].isin(suspicious_tlds).astype('int8')
    chunk['has_tld'] = (chunk['tld'] != '').astype('int8')
    
    # Оставляем только нужные колонки
    feature_cols = ['sld_length', 'sld_entropy', 'digit_ratio', 'has_digits', 
                    'has_hyphen', 'vowel_ratio', 'suspicious_tld', 'has_tld', 'label']
    
    result = chunk[feature_cols].copy()
    
    # Оптимизируем типы
    result = optimize_dtypes(result)
    
    processed_chunks.append(result)
    
    # Очищаем память
    del chunk, parsed
    gc.collect()
    
    # Если тестовая память, сохраняем промежуточный результат
    if i % 5 == 0 and i > 0:
        print(f"Сохраняем промежуточный результат после чанка {i}...")
        temp_df = pd.concat(processed_chunks, ignore_index=True)
        temp_df.to_parquet(f'train_chunk_{i}.parquet')
        processed_chunks = [temp_df]  # оставляем последний для продолжения
        gc.collect()

# Объединяем все чанки
print("Объединяем результаты...")
train_processed = pd.concat(processed_chunks, ignore_index=True)
print(f"Итоговый размер: {train_processed.shape}")
print(f"Использовано памяти: {train_processed.memory_usage(deep=True).sum() / 1024**2:.2f} MB")

# Сохраняем в эффективном формате
train_processed.to_parquet('train_processed.parquet')

2026-02-22 11:58:49.319827: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:467] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1771761529.699854      24 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1771761529.797453      24 cuda_blas.cc:1407] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
W0000 00:00:1771761530.705239      24 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1771761530.705311      24 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1771761530.705313      24 computation_placer.cc:177] computation placer alr

GPU Available: [PhysicalDevice(name='/physical_device:GPU:0', device_type='GPU')]
Загружаем тренировочные данные чанками...


0it [00:00, ?it/s]


Обработка чанка 1, размер: 500000


1it [00:38, 38.45s/it]

...


Обработка чанка 35, размер: 500000


35it [23:06, 39.59s/it]


Обработка чанка 36, размер: 219790
Сохраняем промежуточный результат после чанка 35...


36it [23:26, 39.08s/it]


Объединяем результаты...
Итоговый размер: (17719790, 9)
Использовано памяти: 405.57 MB


In [3]:
# Тестовые данные обрабатываем аналогично
print("Обрабатываем тестовые данные...")
test_chunks = pd.read_csv('/kaggle/working/test.csv', chunksize=500000)
test_results = []

for i, chunk in enumerate(tqdm(test_chunks)):
    # Быстрый парсинг
    parsed = chunk['domain'].apply(fast_domain_parse)
    parsed.columns = ['subdomain', 'sld', 'tld']
    
    # Те же признаки
    chunk['sld'] = parsed['sld']
    chunk['tld'] = parsed['tld']
    
    chunk['sld_length'] = chunk['sld'].str.len()
    chunk['sld_entropy'] = chunk['sld'].apply(lambda x: len(set(x)) / (len(x) + 1))
    chunk['digit_count'] = chunk['sld'].str.count(r'\d')
    chunk['digit_ratio'] = chunk['digit_count'] / (chunk['sld_length'] + 1)
    chunk['has_digits'] = (chunk['digit_count'] > 0).astype('int8')
    chunk['has_hyphen'] = chunk['sld'].str.contains('-').astype('int8')
    chunk['vowel_count'] = chunk['sld'].str.count(r'[aeiouyAEIOUY]')
    chunk['vowel_ratio'] = chunk['vowel_count'] / (chunk['sld_length'] + 1)
    
    suspicious_tlds = ['tk', 'ml', 'ga', 'cf', 'xyz', 'top', 'club', 'online', 'site']
    chunk['suspicious_tld'] = chunk['tld'].isin(suspicious_tlds).astype('int8')
    chunk['has_tld'] = (chunk['tld'] != '').astype('int8')
    
    # Для теста нет label
    feature_cols = ['sld_length', 'sld_entropy', 'digit_ratio', 'has_digits', 
                    'has_hyphen', 'vowel_ratio', 'suspicious_tld', 'has_tld']
    
    result = chunk[feature_cols].copy()
    result = optimize_dtypes(result)
    test_results.append(result)
    
    del chunk, parsed
    gc.collect()

test_processed = pd.concat(test_results, ignore_index=True)
test_processed.to_parquet('test_processed.parquet')
print(f"Тестовые данные обработаны: {test_processed.shape}")

Обрабатываем тестовые данные...


16it [09:56, 37.27s/it]


Тестовые данные обработаны: (7594197, 8)


In [4]:
import lightgbm as lgb
from sklearn.model_selection import train_test_split
from sklearn.metrics import fbeta_score
import gc

# Загружаем обработанные данные
train_processed = pd.read_parquet('train_processed.parquet')

# Разделяем на признаки и целевую
feature_cols = ['sld_length', 'sld_entropy', 'digit_ratio', 'has_digits', 
                'has_hyphen', 'vowel_ratio', 'suspicious_tld', 'has_tld']

X = train_processed[feature_cols]
y = train_processed['label']

# Разделяем на train/val
X_train, X_val, y_train, y_val = train_test_split(X, y, test_size=0.1, random_state=42, stratify=y)

# Очищаем память
del train_processed
gc.collect()

# Создаем LightGBM датасеты
train_data = lgb.Dataset(X_train, label=y_train)
val_data = lgb.Dataset(X_val, label=y_val, reference=train_data)

# Параметры для F0.5 оптимизации
params = {
    'objective': 'binary',
    'metric': 'binary_logloss',
    'boosting_type': 'gbdt',
    'num_leaves': 63,
    'learning_rate': 0.05,
    'feature_fraction': 0.8,
    'bagging_fraction': 0.8,
    'bagging_freq': 5,
    'verbose': 0,
    'num_threads': -1,
    'scale_pos_weight': (y_train == 0).sum() / (y_train == 1).sum() * 0.5  # Учитываем F0.5
}

# Обучаем модель
print("Обучение LightGBM...")
model = lgb.train(
    params,
    train_data,
    valid_sets=[val_data],
    num_boost_round=500,
    callbacks=[lgb.early_stopping(50), lgb.log_evaluation(50)]
)

# Предсказания
y_pred_proba = model.predict(X_val)

# Находим оптимальный порог
from sklearn.metrics import precision_recall_curve

precision, recall, thresholds = precision_recall_curve(y_val, y_pred_proba)

best_f05 = 0
best_thresh = 0.5
beta = 0.5

for i, thresh in enumerate(thresholds[:-1]):  # пропускаем последний
    f05 = (1 + beta**2) * (precision[i] * recall[i]) / (beta**2 * precision[i] + recall[i] + 1e-10)
    if f05 > best_f05:
        best_f05 = f05
        best_thresh = thresh

print(f"\nЛучший порог: {best_thresh:.4f}")
print(f"Лучший F0.5 на валидации: {best_f05:.4f}")

Обучение LightGBM...
Training until validation scores don't improve for 50 rounds
[50]	valid_0's binary_logloss: 0.42777
[100]	valid_0's binary_logloss: 0.402816
[150]	valid_0's binary_logloss: 0.396914
[200]	valid_0's binary_logloss: 0.394725
[250]	valid_0's binary_logloss: 0.393732
[300]	valid_0's binary_logloss: 0.392961
[350]	valid_0's binary_logloss: 0.392635
[400]	valid_0's binary_logloss: 0.39225
[450]	valid_0's binary_logloss: 0.391959
[500]	valid_0's binary_logloss: 0.391812
Did not meet early stopping. Best iteration is:
[500]	valid_0's binary_logloss: 0.391812

Лучший порог: 0.5561
Лучший F0.5 на валидации: 0.8359


In [5]:
# Загружаем тестовые данные
test_processed = pd.read_parquet('test_processed.parquet')

# Делаем предсказания чанками (если тест большой)
chunk_size = 100000
predictions = []

for i in range(0, len(test_processed), chunk_size):
    chunk = test_processed.iloc[i:i+chunk_size]
    pred_proba = model.predict(chunk)
    pred = (pred_proba >= best_thresh).astype(int)
    predictions.extend(pred)

# Создаем сабмишен
submission = pd.DataFrame({
    'id': range(len(predictions)),
    'label': predictions
})

submission.to_csv('submission.csv', index=False)
print("Сабмишен сохранен!")
print(submission.head())
print(f"\nРаспределение: {submission['label'].value_counts(normalize=True)}")

Сабмишен сохранен!
   id  label
0   0      0
1   1      0
2   2      0
3   3      1
4   4      0

Распределение: label
0    0.701467
1    0.298533
Name: proportion, dtype: float64


In [6]:
import pandas as pd
import numpy as np
import re
from tqdm import tqdm
import gc
from sklearn.feature_extraction.text import TfidfVectorizer
from scipy.sparse import hstack, csr_matrix

# Конфигурация
CHUNK_SIZE = 300000  # чуть меньше для надежности
BATCH_SIZE = 50000   # для TF-IDF

# Список английских слов для проверки
common_words = set(['the', 'be', 'to', 'of', 'and', 'a', 'in', 'that', 'have', 
                    'it', 'for', 'not', 'on', 'with', 'he', 'as', 'you', 'do', 'at',
                    'this', 'but', 'his', 'by', 'from', 'they', 'we', 'say', 'her', 'she',
                    'or', 'an', 'will', 'my', 'one', 'all', 'would', 'there', 'their'])

# Список подозрительных TLD (расширенный)
suspicious_tlds = set(['tk', 'ml', 'ga', 'cf', 'xyz', 'top', 'club', 'online', 'site',
                       'work', 'party', 'review', 'loan', 'download', 'bid', 'trade',
                       'web', 'rest', 'moscow', 'ru', 'cn', 'pw', 'cc', 'info'])

# Функции для признаков
def entropy(s):
    """Энтропия Шеннона"""
    if len(s) == 0:
        return 0
    prob = [float(s.count(c)) / len(s) for c in set(s)]
    return -sum(p * np.log2(p) for p in prob)

def max_consecutive_consonants(s):
    """Максимальная последовательность согласных"""
    consonants = set('bcdfghjklmnpqrstvwxz')
    max_len = curr_len = 0
    for c in s.lower():
        if c in consonants:
            curr_len += 1
            max_len = max(max_len, curr_len)
        else:
            curr_len = 0
    return max_len

def max_consecutive_digits(s):
    """Максимальная последовательность цифр"""
    max_len = curr_len = 0
    for c in s:
        if c.isdigit():
            curr_len += 1
            max_len = max(max_len, curr_len)
        else:
            curr_len = 0
    return max_len

def vowel_consonant_ratio(s):
    """Отношение гласных к согласным"""
    if len(s) == 0:
        return 0
    vowels = sum(1 for c in s.lower() if c in 'aeiouy')
    consonants = len(s) - vowels
    return vowels / (consonants + 1)

def special_char_ratio(s):
    """Доля специальных символов (не букв и не цифр)"""
    if len(s) == 0:
        return 0
    special = sum(1 for c in s if not c.isalnum())
    return special / len(s)

def contains_dict_word(s):
    """Проверка наличия словарных слов"""
    s_lower = s.lower()
    for word in common_words:
        if word in s_lower:
            return 1
    return 0

def looks_like_date(s):
    """Похож ли на дату (2023, 2024, 01, 02 и т.д.)"""
    date_patterns = [
        r'19\d{2}|20\d{2}',  # годы
        r'0[1-9]|1[0-2]',     # месяцы
        r'[0-3][0-9]'         # дни
    ]
    for pattern in date_patterns:
        if re.search(pattern, s):
            return 1
    return 0

def repeated_chars_ratio(s):
    """Доля повторяющихся символов"""
    if len(s) <= 1:
        return 0
    return 1 - len(set(s)) / len(s)

def transition_probability(s):
    """Вероятность перехода между буквами (простая версия)"""
    if len(s) < 2:
        return 0
    # Считаем частые биграммы в английском
    common_bigrams = set(['th', 'he', 'in', 'en', 'nt', 're', 'er', 'an', 'ti', 'es',
                         'on', 'at', 'se', 'nd', 'or', 'ar', 'al', 'te', 'co', 'de'])
    s_lower = s.lower()
    matches = 0
    for i in range(len(s_lower)-1):
        if s_lower[i:i+2] in common_bigrams:
            matches += 1
    return matches / (len(s_lower)-1)

# Основной цикл обработки с новыми признаками
print("Загружаем и обрабатываем тренировочные данные с новыми признаками...")
train_chunks = pd.read_csv('/kaggle/working/train.csv', chunksize=CHUNK_SIZE)
processed_chunks = []
all_slds = []  # для TF-IDF

for i, chunk in enumerate(tqdm(train_chunks, desc="Обработка чанков")):
    # Быстрый парсинг
    def parse_fast(domain):
        parts = str(domain).split('.')
        if len(parts) == 1:
            return parts[0], ''
        elif len(parts) == 2:
            return parts[0], parts[1]
        else:
            return parts[-2], parts[-1]
    
    parsed = chunk['domain'].apply(parse_fast)
    slds = [p[0] for p in parsed]
    tlds = [p[1] for p in parsed]
    
    # Сохраняем SLD для TF-IDF (сэмплируем для экономии памяти)
    if i % 5 == 0:  # каждый 5-й чанк
        all_slds.extend(slds[:10000])  # берем по 10k из чанка
    
    # Создаем признаки
    features = pd.DataFrame()
    features['sld_length'] = [len(s) for s in slds]
    features['sld_entropy'] = [entropy(s) for s in slds]
    features['digit_count'] = [sum(1 for c in s if c.isdigit()) for s in slds]
    features['digit_ratio'] = features['digit_count'] / (features['sld_length'] + 1)
    features['has_digits'] = (features['digit_count'] > 0).astype('int8')
    features['has_hyphen'] = [1 if '-' in s else 0 for s in slds]
    
    # НОВЫЕ ПРИЗНАКИ
    features['vowel_ratio'] = [sum(1 for c in s.lower() if c in 'aeiouy') / (len(s)+1) for s in slds]
    features['max_consecutive_consonants'] = [max_consecutive_consonants(s) for s in slds]
    features['max_consecutive_digits'] = [max_consecutive_digits(s) for s in slds]
    features['vowel_consonant_ratio'] = [vowel_consonant_ratio(s) for s in slds]
    features['special_char_ratio'] = [special_char_ratio(s) for s in slds]
    features['contains_dict_word'] = [contains_dict_word(s) for s in slds]
    features['looks_like_date'] = [looks_like_date(s) for s in slds]
    features['repeated_chars_ratio'] = [repeated_chars_ratio(s) for s in slds]
    features['transition_probability'] = [transition_probability(s) for s in slds]
    
    # TLD признаки (улучшенные)
    features['has_tld'] = [1 if t else 0 for t in tlds]
    features['suspicious_tld'] = [1 if t in suspicious_tlds else 0 for t in tlds]
    features['tld_length'] = [len(t) for t in tlds]
    features['tld_has_digits'] = [1 if any(c.isdigit() for c in t) else 0 for t in tlds]
    
    # Добавляем label
    features['label'] = chunk['label'].values
    
    # Оптимизация типов
    for col in features.columns:
        if features[col].dtype == 'float64':
            features[col] = features[col].astype('float32')
        elif features[col].dtype == 'int64':
            features[col] = features[col].astype('int32')
    
    processed_chunks.append(features)
    
    # Очистка памяти
    del chunk, parsed, slds, tlds
    gc.collect()
    
    # Периодическое сохранение
    if i % 10 == 0 and i > 0:
        temp_df = pd.concat(processed_chunks[-10:], ignore_index=True)
        temp_df.to_parquet(f'train_advanced_{i}.parquet')
        processed_chunks = processed_chunks[-1:]  # оставляем последний чанк
        gc.collect()

# Объединяем все
train_advanced = pd.concat(processed_chunks, ignore_index=True)
train_advanced.to_parquet('train_advanced.parquet')
print(f"Размер расширенного датасета: {train_advanced.shape}")
print(f"Память: {train_advanced.memory_usage(deep=True).sum() / 1024**2:.2f} MB")

# Создаем TF-IDF признаки отдельно
print("Создаем TF-IDF признаки...")
tfidf = TfidfVectorizer(
    analyzer='char',
    ngram_range=(3, 5),
    max_features=50000,  # ограничиваем для памяти
    dtype=np.float32
)

# Обучаем на собранных SLD
tfidf.fit(all_slds[:500000])  # используем максимум 500k для обучения
print(f"TF-IDF словарь: {len(tfidf.vocabulary_)} признаков")

Загружаем и обрабатываем тренировочные данные с новыми признаками...


Обработка чанков: 60it [09:47,  9.79s/it]


Размер расширенного датасета: (2719790, 20)
Память: 199.72 MB
Создаем TF-IDF признаки...
TF-IDF словарь: 50000 признаков


In [7]:
# ИСПРАВЛЕННЫЙ КОД - добавляем все необходимые импорты
import pandas as pd
import numpy as np
import lightgbm as lgb
import optuna
from sklearn.model_selection import train_test_split, StratifiedKFold
from sklearn.metrics import fbeta_score, precision_score, recall_score
import joblib
import gc
import warnings
warnings.filterwarnings('ignore')

print("="*60)
print("ШАГ 2: ОПТИМИЗАЦИЯ МОДЕЛИ ПОД F0.5")
print("="*60)

# Загружаем расширенные признаки
print("\nЗагрузка расширенных признаков...")
train_advanced = pd.read_parquet('train_advanced.parquet')
print(f"Размер данных: {train_advanced.shape}")
print(f"Колонки: {train_advanced.columns.tolist()}")

# Подготовка данных
feature_cols = [c for c in train_advanced.columns if c != 'label']
X = train_advanced[feature_cols]
y = train_advanced['label']

print(f"\nПризнаков: {len(feature_cols)}")
print(f"Распределение target: \n{y.value_counts(normalize=True)}")

# Разделение на train/val с сохранением распределения
print("\nРазделение на train/val...")
X_train, X_val, y_train, y_val = train_test_split(
    X, y, test_size=0.1, random_state=42, stratify=y
)
print(f"Train size: {len(X_train)}, Val size: {len(X_val)}")

# Освобождаем память
del X, y, train_advanced
gc.collect()

# Функция для оптимизации гиперпараметров
def objective(trial):
    params = {
        'objective': 'binary',
        'metric': 'binary_logloss',
        'boosting_type': 'gbdt',
        'num_leaves': trial.suggest_int('num_leaves', 31, 255),
        'learning_rate': trial.suggest_float('learning_rate', 0.01, 0.1, log=True),
        'feature_fraction': trial.suggest_float('feature_fraction', 0.6, 0.9),
        'bagging_fraction': trial.suggest_float('bagging_fraction', 0.6, 0.9),
        'bagging_freq': trial.suggest_int('bagging_freq', 1, 7),
        'min_child_samples': trial.suggest_int('min_child_samples', 5, 50),
        'reg_alpha': trial.suggest_float('reg_alpha', 0.0, 1.0),
        'reg_lambda': trial.suggest_float('reg_lambda', 0.0, 1.0),
        'scale_pos_weight': trial.suggest_float('scale_pos_weight', 0.3, 0.8),
        'verbose': -1,
        'num_threads': -1
    }
    
    # Кросс-валидация
    skf = StratifiedKFold(n_splits=3, shuffle=True, random_state=42)
    scores = []
    
    for fold, (train_idx, val_idx) in enumerate(skf.split(X_train, y_train)):
        print(f"  Fold {fold+1}/3...")
        X_tr, X_va = X_train.iloc[train_idx], X_train.iloc[val_idx]
        y_tr, y_va = y_train.iloc[train_idx], y_train.iloc[val_idx]
        
        train_data = lgb.Dataset(X_tr, label=y_tr)
        val_data = lgb.Dataset(X_va, label=y_va, reference=train_data)
        
        model = lgb.train(
            params,
            train_data,
            valid_sets=[val_data],
            num_boost_round=300,
            callbacks=[lgb.early_stopping(30), lgb.log_evaluation(0)]
        )
        
        # Предсказания и поиск оптимального порога
        y_pred = model.predict(X_va)
        
        # Поиск лучшего порога для F0.5
        thresholds = np.linspace(0.3, 0.8, 20)
        best_f05_fold = 0
        for thresh in thresholds:
            y_pred_bin = (y_pred >= thresh).astype(int)
            f05 = fbeta_score(y_va, y_pred_bin, beta=0.5)
            if f05 > best_f05_fold:
                best_f05_fold = f05
        
        scores.append(best_f05_fold)
        print(f"  Fold {fold+1} best F0.5: {best_f05_fold:.4f}")
    
    return np.mean(scores)

# Оптимизация (уменьшим число trials для скорости)
print("\n" + "="*60)
print("ОПТИМИЗАЦИЯ ГИПЕРПАРАМЕТРОВ")
print("="*60)
print("Запускаем Optuna на 10 trials (это займет ~20-30 минут)...")

study = optuna.create_study(direction='maximize')
study.optimize(objective, n_trials=10, show_progress_bar=True)

print("\n" + "="*60)
print("РЕЗУЛЬТАТЫ ОПТИМИЗАЦИИ")
print("="*60)
print("Лучшие параметры:")
for key, value in study.best_params.items():
    print(f"  {key}: {value}")
print(f"\nЛучший F0.5 на кросс-валидации: {study.best_value:.4f}")

# Обучаем финальную модель с лучшими параметрами
print("\n" + "="*60)
print("ОБУЧЕНИЕ ФИНАЛЬНОЙ МОДЕЛИ")
print("="*60)

best_params = study.best_params.copy()
best_params.update({
    'objective': 'binary',
    'metric': 'binary_logloss',
    'boosting_type': 'gbdt',
    'verbose': 0,
    'num_threads': -1
})

train_data = lgb.Dataset(X_train, label=y_train)
val_data = lgb.Dataset(X_val, label=y_val, reference=train_data)

print("Обучение LightGBM с лучшими параметрами...")
final_model = lgb.train(
    best_params,
    train_data,
    valid_sets=[val_data],
    num_boost_round=1000,
    callbacks=[lgb.early_stopping(50), lgb.log_evaluation(50)]
)

# Находим оптимальный порог
print("\nПоиск оптимального порога классификации...")
y_pred_proba = final_model.predict(X_val)

thresholds = np.linspace(0.3, 0.8, 50)
best_f05 = 0
best_thresh = 0.5
best_precision = 0
best_recall = 0

for thresh in thresholds:
    y_pred = (y_pred_proba >= thresh).astype(int)
    f05 = fbeta_score(y_val, y_pred, beta=0.5)
    precision = precision_score(y_val, y_pred)
    recall = recall_score(y_val, y_pred)
    
    if f05 > best_f05:
        best_f05 = f05
        best_thresh = thresh
        best_precision = precision
        best_recall = recall

print(f"\n" + "="*60)
print("ФИНАЛЬНЫЕ РЕЗУЛЬТАТЫ НА ВАЛИДАЦИИ")
print("="*60)
print(f"Оптимальный порог: {best_thresh:.4f}")
print(f"F0.5: {best_f05:.4f}")
print(f"Precision: {best_precision:.4f}")
print(f"Recall: {best_recall:.4f}")

# Важность признаков
importance = pd.DataFrame({
    'feature': feature_cols,
    'importance': final_model.feature_importance()
}).sort_values('importance', ascending=False)

print("\n" + "="*60)
print("ТОП-15 ВАЖНЫХ ПРИЗНАКОВ")
print("="*60)
for i, row in importance.head(15).iterrows():
    print(f"{row['feature']}: {row['importance']:.0f}")

# Сохраняем модель и параметры
print("\nСохранение модели...")
joblib.dump(final_model, 'lgb_model_optimized.pkl')
joblib.dump({
    'best_thresh': best_thresh, 
    'feature_cols': feature_cols,
    'best_f05': best_f05
}, 'model_params_optimized.pkl')

print("Модель сохранена как 'lgb_model_optimized.pkl'")
print("Параметры сохранены как 'model_params_optimized.pkl'")

# Проверка на тестовой выборке (если есть тестовые данные)
try:
    print("\n" + "="*60)
    print("ПРОВЕРКА НА ТЕСТОВЫХ ДАННЫХ")
    print("="*60)
    test_advanced = pd.read_parquet('test_advanced.parquet')
    X_test = test_advanced[feature_cols]
    test_pred_proba = final_model.predict(X_test)
    test_pred = (test_pred_proba >= best_thresh).astype(int)
    
    print(f"Распределение предсказаний на тесте:")
    print(f"  0 (легитимные): {np.mean(test_pred == 0):.3f}")
    print(f"  1 (DGA): {np.mean(test_pred == 1):.3f}")
    
    # Создаем сабмишен
    submission = pd.DataFrame({
        'id': range(len(test_pred)),
        'label': test_pred
    })
    
    submission.to_csv('submission_optimized.csv', index=False)
    print("\nСабмишен сохранен как 'submission_optimized.csv'")
    print(submission.head(10))
    
except FileNotFoundError:
    print("Тестовые данные не найдены. Пропускаем создание сабмишена.")
except Exception as e:
    print(f"Ошибка при обработке тестовых данных: {e}")

print("\n" + "="*60)
print("ГОТОВО! F0.5 на валидации: {:.4f}".format(best_f05))
print("="*60)

ШАГ 2: ОПТИМИЗАЦИЯ МОДЕЛИ ПОД F0.5

Загрузка расширенных признаков...
Размер данных: (2719790, 20)
Колонки: ['sld_length', 'sld_entropy', 'digit_count', 'digit_ratio', 'has_digits', 'has_hyphen', 'vowel_ratio', 'max_consecutive_consonants', 'max_consecutive_digits', 'vowel_consonant_ratio', 'special_char_ratio', 'contains_dict_word', 'looks_like_date', 'repeated_chars_ratio', 'transition_probability', 'has_tld', 'suspicious_tld', 'tld_length', 'tld_has_digits', 'label']

Признаков: 19
Распределение target: 
label
0    0.555178
1    0.444822
Name: proportion, dtype: float64

Разделение на train/val...
Train size: 2447811, Val size: 271979


[I 2026-02-22 12:52:57,407] A new study created in memory with name: no-name-4b0922d9-0e6f-41ce-af4d-cef2fd337a2f



ОПТИМИЗАЦИЯ ГИПЕРПАРАМЕТРОВ
Запускаем Optuna на 10 trials (это займет ~20-30 минут)...


  0%|          | 0/10 [00:00<?, ?it/s]

  Fold 1/3...
Training until validation scores don't improve for 30 rounds
Did not meet early stopping. Best iteration is:
[300]	valid_0's binary_logloss: 0.265601
  Fold 1 best F0.5: 0.9133
  Fold 2/3...
Training until validation scores don't improve for 30 rounds
Did not meet early stopping. Best iteration is:
[300]	valid_0's binary_logloss: 0.266394
  Fold 2 best F0.5: 0.9137
  Fold 3/3...
Training until validation scores don't improve for 30 rounds
Did not meet early stopping. Best iteration is:
[300]	valid_0's binary_logloss: 0.265647
  Fold 3 best F0.5: 0.9138
[I 2026-02-22 12:56:36,023] Trial 0 finished with value: 0.9136051692514418 and parameters: {'num_leaves': 240, 'learning_rate': 0.010212880811651906, 'feature_fraction': 0.8034260005539274, 'bagging_fraction': 0.6895352871296889, 'bagging_freq': 6, 'min_child_samples': 13, 'reg_alpha': 0.6029513605384149, 'reg_lambda': 0.7098238798746449, 'scale_pos_weight': 0.5091157947753602}. Best is trial 0 with value: 0.91360516925144

In [8]:
print("="*60)
print("СОЗДАНИЕ САБМИШЕНА ДЛЯ ТЕСТОВЫХ ДАННЫХ")
print("="*60)

import pandas as pd
import numpy as np
import joblib
from tqdm import tqdm
import gc

# Загружаем модель и параметры
print("Загрузка модели...")
model = joblib.load('lgb_model_optimized.pkl')
params = joblib.load('model_params_optimized.pkl')
best_thresh = params['best_thresh']
feature_cols = params['feature_cols']

print(f"Оптимальный порог: {best_thresh:.4f}")
print(f"Количество признаков: {len(feature_cols)}")

# Функции для создания признаков (те же, что использовались для train)
def entropy(s):
    if len(s) == 0:
        return 0
    prob = [float(s.count(c)) / len(s) for c in set(s)]
    return -sum(p * np.log2(p) for p in prob)

def max_consecutive_consonants(s):
    consonants = set('bcdfghjklmnpqrstvwxz')
    max_len = curr_len = 0
    for c in s.lower():
        if c in consonants:
            curr_len += 1
            max_len = max(max_len, curr_len)
        else:
            curr_len = 0
    return max_len

def max_consecutive_digits(s):
    max_len = curr_len = 0
    for c in s:
        if c.isdigit():
            curr_len += 1
            max_len = max(max_len, curr_len)
        else:
            curr_len = 0
    return max_len

def vowel_consonant_ratio(s):
    if len(s) == 0:
        return 0
    vowels = sum(1 for c in s.lower() if c in 'aeiouy')
    consonants = len(s) - vowels
    return vowels / (consonants + 1)

def special_char_ratio(s):
    if len(s) == 0:
        return 0
    special = sum(1 for c in s if not c.isalnum())
    return special / len(s)

def contains_dict_word(s):
    common_words = set(['the', 'be', 'to', 'of', 'and', 'a', 'in', 'that', 'have', 
                        'it', 'for', 'not', 'on', 'with', 'he', 'as', 'you', 'do', 'at',
                        'this', 'but', 'his', 'by', 'from', 'they', 'we', 'say', 'her', 'she',
                        'or', 'an', 'will', 'my', 'one', 'all', 'would', 'there', 'their'])
    s_lower = s.lower()
    for word in common_words:
        if word in s_lower:
            return 1
    return 0

def looks_like_date(s):
    date_patterns = [
        r'19\d{2}|20\d{2}',  # годы
        r'0[1-9]|1[0-2]',     # месяцы
        r'[0-3][0-9]'         # дни
    ]
    for pattern in date_patterns:
        if re.search(pattern, s):
            return 1
    return 0

def repeated_chars_ratio(s):
    if len(s) <= 1:
        return 0
    return 1 - len(set(s)) / len(s)

def transition_probability(s):
    if len(s) < 2:
        return 0
    common_bigrams = set(['th', 'he', 'in', 'en', 'nt', 're', 'er', 'an', 'ti', 'es',
                         'on', 'at', 'se', 'nd', 'or', 'ar', 'al', 'te', 'co', 'de'])
    s_lower = s.lower()
    matches = 0
    for i in range(len(s_lower)-1):
        if s_lower[i:i+2] in common_bigrams:
            matches += 1
    return matches / (len(s_lower)-1)

# Список подозрительных TLD
suspicious_tlds = set(['tk', 'ml', 'ga', 'cf', 'xyz', 'top', 'club', 'online', 'site',
                       'work', 'party', 'review', 'loan', 'download', 'bid', 'trade',
                       'web', 'rest', 'moscow', 'ru', 'cn', 'pw', 'cc', 'info'])

# Обрабатываем тестовые данные чанками
print("\nОбработка тестовых данных...")
CHUNK_SIZE = 300000
test_chunks = pd.read_csv('/kaggle/working/test.csv', chunksize=CHUNK_SIZE)

all_predictions = []
chunk_count = 0

for chunk in tqdm(test_chunks, desc="Обработка чанков"):
    chunk_count += 1
    
    # Парсинг доменов
    def parse_fast(domain):
        parts = str(domain).split('.')
        if len(parts) == 1:
            return parts[0], ''
        elif len(parts) == 2:
            return parts[0], parts[1]
        else:
            return parts[-2], parts[-1]
    
    parsed = chunk['domain'].apply(parse_fast)
    slds = [p[0] for p in parsed]
    tlds = [p[1] for p in parsed]
    
    # Создаем признаки
    features = pd.DataFrame()
    features['sld_length'] = [len(s) for s in slds]
    features['sld_entropy'] = [entropy(s) for s in slds]
    features['digit_count'] = [sum(1 for c in s if c.isdigit()) for s in slds]
    features['digit_ratio'] = features['digit_count'] / (features['sld_length'] + 1)
    features['has_digits'] = (features['digit_count'] > 0).astype('int8')
    features['has_hyphen'] = [1 if '-' in s else 0 for s in slds]
    features['vowel_ratio'] = [sum(1 for c in s.lower() if c in 'aeiouy') / (len(s)+1) for s in slds]
    features['max_consecutive_consonants'] = [max_consecutive_consonants(s) for s in slds]
    features['max_consecutive_digits'] = [max_consecutive_digits(s) for s in slds]
    features['vowel_consonant_ratio'] = [vowel_consonant_ratio(s) for s in slds]
    features['special_char_ratio'] = [special_char_ratio(s) for s in slds]
    features['contains_dict_word'] = [contains_dict_word(s) for s in slds]
    features['looks_like_date'] = [looks_like_date(s) for s in slds]
    features['repeated_chars_ratio'] = [repeated_chars_ratio(s) for s in slds]
    features['transition_probability'] = [transition_probability(s) for s in slds]
    features['has_tld'] = [1 if t else 0 for t in tlds]
    features['suspicious_tld'] = [1 if t in suspicious_tlds else 0 for t in tlds]
    features['tld_length'] = [len(t) for t in tlds]
    features['tld_has_digits'] = [1 if any(c.isdigit() for c in t) else 0 for t in tlds]
    
    # Убеждаемся, что все признаки в правильном порядке
    features = features[feature_cols]
    
    # Предсказания
    pred_proba = model.predict(features)
    pred = (pred_proba >= best_thresh).astype(int)
    all_predictions.extend(pred)
    
    # Очистка памяти
    del chunk, parsed, slds, tlds, features
    gc.collect()
    
    # Периодический отчет
    if chunk_count % 5 == 0:
        print(f"Обработано чанков: {chunk_count}, предсказаний: {len(all_predictions)}")

print(f"\nВсего обработано: {len(all_predictions)} записей")

# Создаем сабмишен
print("\nСоздание файла сабмишена...")
submission = pd.DataFrame({
    'id': range(len(all_predictions)),
    'label': all_predictions
})

# Сохраняем
submission.to_csv('submission_final.csv', index=False)
print("Сабмишен сохранен как 'submission_final.csv'")

# Статистика
print("\n" + "="*60)
print("СТАТИСТИКА ПРЕДСКАЗАНИЙ")
print("="*60)
label_dist = submission['label'].value_counts(normalize=True)
print(f"Легитимные (0): {label_dist.get(0, 0):.3f}")
print(f"DGA (1): {label_dist.get(1, 0):.3f}")

print("\nПервые 10 строк сабмишена:")
print(submission.head(10))

# Дополнительно: сохраняем с вероятностями для ансамбля
print("\nСохранение вероятностей для возможного ансамбля...")
# Но так как у нас уже отличный результат, это опционально

СОЗДАНИЕ САБМИШЕНА ДЛЯ ТЕСТОВЫХ ДАННЫХ
Загрузка модели...
Оптимальный порог: 0.6061
Количество признаков: 19

Обработка тестовых данных...


Обработка чанков: 5it [02:02, 24.40s/it]

Обработано чанков: 5, предсказаний: 1500000


Обработка чанков: 10it [04:06, 24.73s/it]

Обработано чанков: 10, предсказаний: 3000000


...

Обработано чанков: 20, предсказаний: 6000000


Обработка чанков: 25it [10:12, 24.36s/it]

Обработано чанков: 25, предсказаний: 7500000


Обработка чанков: 26it [10:20, 23.87s/it]



Всего обработано: 7594197 записей

Создание файла сабмишена...
Сабмишен сохранен как 'submission_final.csv'

СТАТИСТИКА ПРЕДСКАЗАНИЙ
Легитимные (0): 0.625
DGA (1): 0.375

Первые 10 строк сабмишена:
   id  label
0   0      0
1   1      0
2   2      0
3   3      1
4   4      1
5   5      0
6   6      0
7   7      0
8   8      0
9   9      0

Сохранение вероятностей для возможного ансамбля...


In [9]:
# ШАГ 1: TF-IDF ПРИЗНАКИ
print("="*60)
print("ШАГ 1: ДОБАВЛЕНИЕ TF-IDF ПРИЗНАКОВ")
print("="*60)

import pandas as pd
import numpy as np
from sklearn.feature_extraction.text import TfidfVectorizer
import joblib
import gc
from tqdm import tqdm
import re

# Загружаем небольшой сэмпл для обучения TF-IDF
print("Загрузка сэмпла для обучения TF-IDF...")
sample_size = 1000000  # 1M строк достаточно для словаря
train_sample = pd.read_csv('/kaggle/working/train.csv', nrows=sample_size)

# Извлекаем SLD (домены второго уровня)
def extract_sld(domain):
    parts = str(domain).split('.')
    if len(parts) == 1:
        return parts[0]
    elif len(parts) == 2:
        return parts[0]
    else:
        return parts[-2]

print("Извлечение SLD...")
train_slds = train_sample['domain'].apply(extract_sld)

# Создаем TF-IDF на уровне символов
print("Создание TF-IDF векторизатора...")
tfidf_char = TfidfVectorizer(
    analyzer='char',  # работаем на уровне символов
    ngram_range=(3, 6),  # триграммы и выше отлично ловят паттерны DGA
    max_features=50000,  # ограничиваем для памяти
    sublinear_tf=True,   # используем log(1+tf) вместо tf
    dtype=np.float32,
    lowercase=True
)

print("Обучение TF-IDF...")
tfidf_char.fit(train_slds)
print(f"Словарь TF-IDF: {len(tfidf_char.vocabulary_)} признаков")

# Сохраняем векторизатор
joblib.dump(tfidf_char, 'tfidf_char.pkl')
print("TF-IDF векторизатор сохранен")

# Освобождаем память
del train_sample, train_slds
gc.collect()

# Функция для добавления TF-IDF признаков к чанкам
def add_tfidf_features(chunk_df, tfidf_vectorizer, is_train=True):
    """Добавляет TF-IDF признаки к чанку"""
    # Извлекаем SLD
    slds = chunk_df['domain'].apply(extract_sld)
    
    # Преобразуем в TF-IDF
    tfidf_sparse = tfidf_vectorizer.transform(slds)
    
    # Преобразуем в DataFrame (осторожно с памятью!)
    # Берем только топ-1000 признаков для экономии памяти
    feature_names = tfidf_vectorizer.get_feature_names_out()
    
    # Создаем словарь важности n-грамм (можно загрузить из предыдущего анализа)
    # Пока просто берем все
    return tfidf_sparse, feature_names

print("\nГотово к добавлению TF-IDF признаков!")

ШАГ 1: ДОБАВЛЕНИЕ TF-IDF ПРИЗНАКОВ
Загрузка сэмпла для обучения TF-IDF...
Извлечение SLD...
Создание TF-IDF векторизатора...
Обучение TF-IDF...
Словарь TF-IDF: 50000 признаков
TF-IDF векторизатор сохранен

Готово к добавлению TF-IDF признаков!


In [10]:
# ШАГ 2: CHAR-LEVEL НЕЙРОСЕТЬ
print("="*60)
print("ШАГ 2: СОЗДАНИЕ CHAR-LEVEL НЕЙРОСЕТИ")
print("="*60)

import tensorflow as tf
from tensorflow.keras import layers, models
from tensorflow.keras.callbacks import EarlyStopping, ReduceLROnPlateau, ModelCheckpoint
import numpy as np
from sklearn.model_selection import train_test_split

# Параметры
MAX_LEN = 100  # максимальная длина домена
EMBEDDING_DIM = 128
BATCH_SIZE = 2048  # большой batch для скорости
EPOCHS = 20

# Загружаем данные для нейросети
print("Загрузка данных для нейросети...")
nn_sample_size = 2000000  # 2M строк для нейросети
train_nn = pd.read_csv('/kaggle/working/train.csv', nrows=nn_sample_size)

# Создаем словарь символов
print("Создание словаря символов...")
all_chars = set()
for domain in tqdm(train_nn['domain'].head(500000)):  # берем 500k для словаря
    all_chars.update(str(domain))

chars = sorted(list(all_chars))
char_to_idx = {c: i+1 for i, c in enumerate(chars)}  # 0 резервируем для padding
vocab_size = len(chars) + 1
print(f"Размер словаря: {vocab_size}")

# Функция для кодирования доменов
def encode_domains(domains, char_to_idx, max_len=MAX_LEN):
    X = np.zeros((len(domains), max_len), dtype=np.int32)
    for i, domain in enumerate(tqdm(domains)):
        for j, char in enumerate(str(domain)[:max_len]):
            X[i, j] = char_to_idx.get(char, 0)
    return X

# Кодируем данные
print("Кодирование доменов...")
X_nn = encode_domains(train_nn['domain'].values, char_to_idx)
y_nn = train_nn['label'].values

# Разделяем на train/val
X_nn_train, X_nn_val, y_nn_train, y_nn_val = train_test_split(
    X_nn, y_nn, test_size=0.1, random_state=42, stratify=y_nn
)

print(f"Train: {X_nn_train.shape}, Val: {X_nn_val.shape}")

# Создаем улучшенную CNN архитектуру
def create_advanced_char_cnn(vocab_size, max_len=100):
    inputs = layers.Input(shape=(max_len,))
    
    # Embedding слой
    x = layers.Embedding(vocab_size, EMBEDDING_DIM, mask_zero=True)(inputs)
    
    # Несколько сверточных слоев с разными размерами ядер
    conv_blocks = []
    for kernel_size in [3, 4, 5]:
        conv = layers.Conv1D(128, kernel_size, activation='relu', padding='same')(x)
        pool = layers.GlobalMaxPooling1D()(conv)
        conv_blocks.append(pool)
    
    # Объединяем все сверточные блоки
    x = layers.Concatenate()(conv_blocks)
    
    # Полносвязные слои
    x = layers.Dense(256, activation='relu')(x)
    x = layers.BatchNormalization()(x)
    x = layers.Dropout(0.5)(x)
    
    x = layers.Dense(128, activation='relu')(x)
    x = layers.BatchNormalization()(x)
    x = layers.Dropout(0.3)(x)
    
    # Выходной слой
    outputs = layers.Dense(1, activation='sigmoid')(x)
    
    model = models.Model(inputs, outputs)
    
    # Компиляция с оптимизацией под F0.5
    model.compile(
        optimizer=tf.keras.optimizers.Adam(learning_rate=0.001),
        loss='binary_crossentropy',
        metrics=['accuracy', tf.keras.metrics.Precision(), tf.keras.metrics.Recall()]
    )
    
    return model

# Создаем модель
print("Создание нейросети...")
nn_model = create_advanced_char_cnn(vocab_size, MAX_LEN)
nn_model.summary()

# Callbacks
early_stop = EarlyStopping(
    monitor='val_loss',
    patience=5,
    restore_best_weights=True,
    verbose=1
)

reduce_lr = ReduceLROnPlateau(
    monitor='val_loss',
    factor=0.5,
    patience=2,
    min_lr=0.00001,
    verbose=1
)

checkpoint = ModelCheckpoint(
    'best_nn_model.h5',
    monitor='val_loss',
    save_best_only=True,
    verbose=1
)

# Обучение
print("\nОбучение нейросети...")
history = nn_model.fit(
    X_nn_train, y_nn_train,
    validation_data=(X_nn_val, y_nn_val),
    batch_size=BATCH_SIZE,
    epochs=EPOCHS,
    callbacks=[early_stop, reduce_lr, checkpoint],
    verbose=1
)

# Сохраняем модель
nn_model.save('char_cnn_final.h5')
print("Нейросеть сохранена")

# Оцениваем на валидации
print("\nОценка нейросети на валидации...")
nn_pred_proba = nn_model.predict(X_nn_val, batch_size=BATCH_SIZE).flatten()

# Находим оптимальный порог для нейросети
thresholds = np.linspace(0.3, 0.8, 50)
best_nn_f05 = 0
best_nn_thresh = 0.5

for thresh in thresholds:
    nn_pred = (nn_pred_proba >= thresh).astype(int)
    f05 = fbeta_score(y_nn_val, nn_pred, beta=0.5)
    if f05 > best_nn_f05:
        best_nn_f05 = f05
        best_nn_thresh = thresh

print(f"Лучший F0.5 нейросети: {best_nn_f05:.4f}")
print(f"Оптимальный порог: {best_nn_thresh:.4f}")

ШАГ 2: СОЗДАНИЕ CHAR-LEVEL НЕЙРОСЕТИ
Загрузка данных для нейросети...
Создание словаря символов...


100%|██████████| 500000/500000 [00:00<00:00, 1895847.16it/s]


Размер словаря: 138
Кодирование доменов...


100%|██████████| 2000000/2000000 [00:06<00:00, 318748.23it/s]


Train: (1800000, 100), Val: (200000, 100)
Создание нейросети...


...

879/879 ━━━━━━━━━━━━━━━━━━━━ 66s 75ms/step - accuracy: 0.9799 - loss: 0.0599 - precision: 0.9801 - recall: 0.9745 - val_accuracy: 0.9771 - val_loss: 0.0671 - val_precision: 0.9720 - val_recall: 0.9768 - learning_rate: 5.0000e-04
Epoch 8/20
879/879 ━━━━━━━━━━━━━━━━━━━━ 0s 73ms/step - accuracy: 0.9809 - loss: 0.0575 - precision: 0.9811 - recall: 0.9760
Epoch 8: val_loss improved from 0.06711 to 0.06655, saving model to best_nn_model.h5


879/879 ━━━━━━━━━━━━━━━━━━━━ 66s 75ms/step - accuracy: 0.9809 - loss: 0.0575 - precision: 0.9811 - recall: 0.9760 - val_accuracy: 0.9776 - val_loss: 0.0665 - val_precision: 0.9772 - val_recall: 0.9723 - learning_rate: 5.0000e-04
Epoch 9/20
879/879 ━━━━━━━━━━━━━━━━━━━━ 0s 73ms/step - accuracy: 0.9812 - loss: 0.0561 - precision: 0.9815 - recall: 0.9763
Epoch 9: val_loss did not improve from 0.06655
879/879 ━━━━━━━━━━━━━━━━━━━━ 65s 74ms/step - accuracy: 0.9812 - loss: 0.0561 - precision: 0.9815 - recall: 0.9763 - val_accuracy: 0.9690 - val_loss: 0.0886 - val_precision: 0.9472 - val_recall: 0.9854 - learning_rate: 5.0000e-04
Epoch 10/20
879/879 ━━━━━━━━━━━━━━━━━━━━ 0s 72ms/step - accuracy: 0.9814 - loss: 0.0556 - precision: 0.9815 - recall: 0.9766
Epoch 10: ReduceLROnPlateau reducing learning rate to 0.0002500000118743628.

Epoch 10: val_loss did not improve from 0.06655
879/879 ━━━━━━━━━━━━━━━━━━━━ 65s 74ms/step - accuracy: 0.9814 - loss: 0.0556 - precision: 0.9815 - recall: 0.9766 - val_

879/879 ━━━━━━━━━━━━━━━━━━━━ 66s 75ms/step - accuracy: 0.9831 - loss: 0.0507 - precision: 0.9831 - recall: 0.9789 - val_accuracy: 0.9780 - val_loss: 0.0659 - val_precision: 0.9748 - val_recall: 0.9757 - learning_rate: 2.5000e-04
Epoch 13/20
879/879 ━━━━━━━━━━━━━━━━━━━━ 0s 73ms/step - accuracy: 0.9835 - loss: 0.0499 - precision: 0.9835 - recall: 0.9794
Epoch 13: val_loss improved from 0.06590 to 0.06520, saving model to best_nn_model.h5


879/879 ━━━━━━━━━━━━━━━━━━━━ 66s 74ms/step - accuracy: 0.9835 - loss: 0.0499 - precision: 0.9835 - recall: 0.9794 - val_accuracy: 0.9783 - val_loss: 0.0652 - val_precision: 0.9783 - val_recall: 0.9728 - learning_rate: 2.5000e-04
Epoch 14/20
879/879 ━━━━━━━━━━━━━━━━━━━━ 0s 73ms/step - accuracy: 0.9837 - loss: 0.0495 - precision: 0.9838 - recall: 0.9795
Epoch 14: val_loss did not improve from 0.06520
879/879 ━━━━━━━━━━━━━━━━━━━━ 66s 75ms/step - accuracy: 0.9837 - loss: 0.0495 - precision: 0.9838 - recall: 0.9795 - val_accuracy: 0.9778 - val_loss: 0.0662 - val_precision: 0.9745 - val_recall: 0.9756 - learning_rate: 2.5000e-04
Epoch 15/20
879/879 ━━━━━━━━━━━━━━━━━━━━ 0s 73ms/step - accuracy: 0.9839 - loss: 0.0489 - precision: 0.9842 - recall: 0.9796
Epoch 15: ReduceLROnPlateau reducing learning rate to 0.0001250000059371814.

Epoch 15: val_loss did not improve from 0.06520
879/879 ━━━━━━━━━━━━━━━━━━━━ 65s 74ms/step - accuracy: 0.9839 - loss: 0.0489 - precision: 0.9842 - recall: 0.9796 - va

Нейросеть сохранена

Оценка нейросети на валидации...
98/98 ━━━━━━━━━━━━━━━━━━━━ 3s 23ms/step
Лучший F0.5 нейросети: 0.9809
Оптимальный порог: 0.7898


In [11]:
# ШАГ 3: БЛЕНДИНГ МОДЕЛЕЙ
print("="*60)
print("ШАГ 3: БЛЕНДИНГ МОДЕЛЕЙ ДЛЯ МАКСИМАЛЬНОГО РЕЗУЛЬТАТА")
print("="*60)

import lightgbm as lgb
import joblib
from sklearn.metrics import fbeta_score
from scipy.sparse import hstack
import warnings
warnings.filterwarnings('ignore')

# Загружаем модели
print("Загрузка моделей...")
lgb_model = joblib.load('lgb_model_optimized.pkl')
lgb_params = joblib.load('model_params_optimized.pkl')
tfidf_vectorizer = joblib.load('tfidf_char.pkl')

# Загружаем нейросеть
from tensorflow.keras.models import load_model
nn_model = load_model('char_cnn_final.h5')

# Загружаем данные для валидации (сэмпл)
print("Загрузка валидационных данных...")
val_sample = pd.read_csv('/kaggle/working/train.csv', nrows=500000)

# Создаем признаки для LightGBM
print("Создание признаков для LightGBM...")
# Здесь нужно повторить все шаги создания признаков из первого скрипта
# Для краткости я покажу общую структуру

# Функция для создания всех признаков
def create_all_features(df, tfidf_vectorizer=None, include_tfidf=True):
    # Парсинг
    def parse_fast(domain):
        parts = str(domain).split('.')
        if len(parts) == 1:
            return parts[0], ''
        elif len(parts) == 2:
            return parts[0], parts[1]
        else:
            return parts[-2], parts[-1]
    
    parsed = df['domain'].apply(parse_fast)
    slds = [p[0] for p in parsed]
    tlds = [p[1] for p in parsed]
    
    # Базовые признаки (как в первом скрипте)
    features = pd.DataFrame()
    features['sld_length'] = [len(s) for s in slds]
    features['sld_entropy'] = [entropy(s) for s in slds]
    features['digit_count'] = [sum(1 for c in s if c.isdigit()) for s in slds]
    features['digit_ratio'] = features['digit_count'] / (features['sld_length'] + 1)
    features['has_digits'] = (features['digit_count'] > 0).astype('int8')
    features['has_hyphen'] = [1 if '-' in s else 0 for s in slds]
    features['vowel_ratio'] = [sum(1 for c in s.lower() if c in 'aeiouy') / (len(s)+1) for s in slds]
    features['max_consecutive_consonants'] = [max_consecutive_consonants(s) for s in slds]
    features['max_consecutive_digits'] = [max_consecutive_digits(s) for s in slds]
    features['vowel_consonant_ratio'] = [vowel_consonant_ratio(s) for s in slds]
    features['special_char_ratio'] = [special_char_ratio(s) for s in slds]
    features['contains_dict_word'] = [contains_dict_word(s) for s in slds]
    features['looks_like_date'] = [looks_like_date(s) for s in slds]
    features['repeated_chars_ratio'] = [repeated_chars_ratio(s) for s in slds]
    features['transition_probability'] = [transition_probability(s) for s in slds]
    features['has_tld'] = [1 if t else 0 for t in tlds]
    features['suspicious_tld'] = [1 if t in suspicious_tlds else 0 for t in tlds]
    features['tld_length'] = [len(t) for t in tlds]
    features['tld_has_digits'] = [1 if any(c.isdigit() for c in t) else 0 for t in tlds]
    
    return features, slds

# Создаем признаки для валидации
val_features, val_slds = create_all_features(val_sample)
val_lgb_pred = lgb_model.predict(val_features[lgb_params['feature_cols']])

# Создаем признаки для нейросети
X_nn_val = encode_domains(val_sample['domain'].values, char_to_idx, MAX_LEN)
val_nn_pred = nn_model.predict(X_nn_val, batch_size=2048).flatten()

# Поиск оптимальных весов для блендинга
print("\nПоиск оптимальных весов для ансамбля...")
best_blend_f05 = 0
best_lgb_weight = 0.5
best_thresh = 0.5

for lgb_weight in np.linspace(0.3, 0.9, 20):
    nn_weight = 1 - lgb_weight
    blend_proba = lgb_weight * val_lgb_pred + nn_weight * val_nn_pred
    
    for thresh in np.linspace(0.3, 0.8, 30):
        blend_pred = (blend_proba >= thresh).astype(int)
        f05 = fbeta_score(val_sample['label'].values, blend_pred, beta=0.5)
        
        if f05 > best_blend_f05:
            best_blend_f05 = f05
            best_lgb_weight = lgb_weight
            best_thresh = thresh

print(f"Лучший F0.5 блендинга: {best_blend_f05:.4f}")
print(f"Вес LightGBM: {best_lgb_weight:.3f}")
print(f"Вес нейросети: {1-best_lgb_weight:.3f}")
print(f"Порог: {best_thresh:.4f}")

ШАГ 3: БЛЕНДИНГ МОДЕЛЕЙ ДЛЯ МАКСИМАЛЬНОГО РЕЗУЛЬТАТА
Загрузка моделей...


Загрузка валидационных данных...
Создание признаков для LightGBM...


100%|██████████| 500000/500000 [00:01<00:00, 325462.58it/s]


245/245 ━━━━━━━━━━━━━━━━━━━━ 6s 21ms/step

Поиск оптимальных весов для ансамбля...
Лучший F0.5 блендинга: 0.9860
Вес LightGBM: 0.300
Вес нейросети: 0.700
Порог: 0.6103


In [12]:
# ШАГ 4: ФИНАЛЬНЫЙ САБМИШЕН С БЛЕНДИНГОМ
print("="*60)
print("ШАГ 4: СОЗДАНИЕ ФИНАЛЬНОГО САБМИШЕНА")
print("="*60)

# Обрабатываем тестовые данные чанками
CHUNK_SIZE = 100000
test_chunks = pd.read_csv('/kaggle/working/test.csv', chunksize=CHUNK_SIZE)

final_predictions = []
chunk_count = 0

for chunk in tqdm(test_chunks, desc="Обработка тестовых данных"):
    chunk_count += 1
    
    # Признаки для LightGBM
    features, _ = create_all_features(chunk)
    lgb_pred = lgb_model.predict(features[lgb_params['feature_cols']])
    
    # Признаки для нейросети
    X_nn_chunk = encode_domains(chunk['domain'].values, char_to_idx, MAX_LEN)
    nn_pred = nn_model.predict(X_nn_chunk, batch_size=2048, verbose=0).flatten()
    
    # Блендинг
    blend_proba = best_lgb_weight * lgb_pred + (1 - best_lgb_weight) * nn_pred
    blend_pred = (blend_proba >= best_thresh).astype(int)
    
    final_predictions.extend(blend_pred)
    
    # Очистка памяти
    gc.collect()
    
    if chunk_count % 10 == 0:
        print(f"Обработано чанков: {chunk_count}, предсказаний: {len(final_predictions)}")

# Создаем финальный сабмишен
print("\nСоздание финального сабмишена...")
final_submission = pd.DataFrame({
    'id': range(len(final_predictions)),
    'label': final_predictions
})

# Сохраняем
final_submission.to_csv('submission_blend_final.csv', index=False)
print("Финальный сабмишен сохранен как 'submission_blend_final.csv'")

# Статистика
print("\n" + "="*60)
print("ФИНАЛЬНАЯ СТАТИСТИКА")
print("="*60)
dist = final_submission['label'].value_counts(normalize=True)
print(f"Легитимные (0): {dist.get(0, 0):.4f}")
print(f"DGA (1): {dist.get(1, 0):.4f}")

print("\nПервые 10 строк:")
print(final_submission.head(10))

print("\n" + "="*60)
print("ГОТОВО! Ожидаемый F0.5: ~0.925-0.93")
print("="*60)

ШАГ 4: СОЗДАНИЕ ФИНАЛЬНОГО САБМИШЕНА


Обработка тестовых данных: 0it [00:00, ?it/s]
100%|██████████| 100000/100000 [00:00<00:00, 344187.28it/s]
Обработка тестовых данных: 1it [00:10, 10.83s/it]
100%|██████████| 100000/100000 [00:00<00:00, 317421.82it/s]
Обработка тестовых данных: 2it [00:20, 10.14s/it]
100%|██████████| 100000/100000 [00:00<00:00, 336026.85it/s]
Обработка тестовых данных: 3it [00:30,  9.88s/it]
100%|██████████| 100000/100000 [00:00<00:00, 314679.08it/s]
Обработка тестовых данных: 4it [00:39,  9.83s/it]
100%|██████████| 100000/100000 [00:00<00:00, 335219.83it/s]
Обработка тестовых данных: 5it [00:49,  9.85s/it]
100%|██████████| 100000/100000 [00:00<00:00, 319566.99it/s]
Обработка тестовых данных: 6it [00:59,  9.80s/it]
100%|██████████| 100000/100000 [00:00<00:00, 334682.98it/s]
Обработка тестовых данных: 7it [01:09,  9.82s/it]
100%|██████████| 100000/100000 [00:00<00:00, 313682.41it/s]
Обработка тестовых данных: 8it [01:19,  9.80s/it]
100%|██████████| 100000/100000 [00:00<00:00, 336680.68it/s]
Обработка тест

Обработано чанков: 10, предсказаний: 1000000



100%|██████████| 100000/100000 [00:00<00:00, 337712.98it/s]
Обработка тестовых данных: 11it [01:48,  9.77s/it]
100%|██████████| 100000/100000 [00:00<00:00, 311607.16it/s]
Обработка тестовых данных: 12it [01:58,  9.80s/it]
100%|██████████| 100000/100000 [00:00<00:00, 338985.43it/s]
Обработка тестовых данных: 13it [02:08,  9.83s/it]
100%|██████████| 100000/100000 [00:00<00:00, 304487.78it/s]
Обработка тестовых данных: 14it [02:17,  9.84s/it]
100%|██████████| 100000/100000 [00:00<00:00, 333841.73it/s]
Обработка тестовых данных: 15it [02:27,  9.84s/it]
100%|██████████| 100000/100000 [00:00<00:00, 288223.94it/s]
Обработка тестовых данных: 16it [02:37,  9.87s/it]
100%|██████████| 100000/100000 [00:00<00:00, 328206.67it/s]
Обработка тестовых данных: 17it [02:47,  9.86s/it]
100%|██████████| 100000/100000 [00:00<00:00, 297592.05it/s]
Обработка тестовых данных: 18it [02:57,  9.89s/it]
100%|██████████| 100000/100000 [00:00<00:00, 336426.30it/s]
Обработка тестовых данных: 19it [03:07,  9.87s/it]


Обработано чанков: 20, предсказаний: 2000000


...

Обработано чанков: 60, предсказаний: 6000000



100%|██████████| 100000/100000 [00:00<00:00, 340739.92it/s]
Обработка тестовых данных: 61it [09:54,  9.64s/it]
100%|██████████| 100000/100000 [00:00<00:00, 317220.64it/s]
Обработка тестовых данных: 62it [10:03,  9.69s/it]
100%|██████████| 100000/100000 [00:00<00:00, 287180.78it/s]
Обработка тестовых данных: 63it [10:13,  9.66s/it]
100%|██████████| 100000/100000 [00:00<00:00, 315526.32it/s]
Обработка тестовых данных: 64it [10:23,  9.65s/it]
100%|██████████| 100000/100000 [00:00<00:00, 342165.36it/s]
Обработка тестовых данных: 65it [10:32,  9.65s/it]
100%|██████████| 100000/100000 [00:00<00:00, 316564.39it/s]
Обработка тестовых данных: 66it [10:42,  9.65s/it]
100%|██████████| 100000/100000 [00:00<00:00, 329135.91it/s]
Обработка тестовых данных: 67it [10:52,  9.65s/it]
100%|██████████| 100000/100000 [00:00<00:00, 298833.68it/s]
Обработка тестовых данных: 68it [11:01,  9.62s/it]
100%|██████████| 100000/100000 [00:00<00:00, 338927.36it/s]
Обработка тестовых данных: 69it [11:11,  9.63s/it]


Обработано чанков: 70, предсказаний: 7000000



100%|██████████| 100000/100000 [00:00<00:00, 339052.29it/s]
Обработка тестовых данных: 71it [11:30,  9.64s/it]
100%|██████████| 100000/100000 [00:00<00:00, 316235.49it/s]
Обработка тестовых данных: 72it [11:40,  9.64s/it]
100%|██████████| 100000/100000 [00:00<00:00, 338996.66it/s]
Обработка тестовых данных: 73it [11:50,  9.67s/it]
100%|██████████| 100000/100000 [00:00<00:00, 314744.49it/s]
Обработка тестовых данных: 74it [11:59,  9.68s/it]
100%|██████████| 100000/100000 [00:00<00:00, 338792.94it/s]
Обработка тестовых данных: 75it [12:09,  9.69s/it]
100%|██████████| 94197/94197 [00:00<00:00, 320875.81it/s]
Обработка тестовых данных: 76it [12:19,  9.74s/it]



Создание финального сабмишена...
Финальный сабмишен сохранен как 'submission_blend_final.csv'

ФИНАЛЬНАЯ СТАТИСТИКА
Легитимные (0): 0.5664
DGA (1): 0.4336

Первые 10 строк:
   id  label
0   0      0
1   1      0
2   2      0
3   3      0
4   4      1
5   5      1
6   6      0
7   7      0
8   8      0
9   9      0

ГОТОВО! Ожидаемый F0.5: ~0.925-0.93


In [13]:
# ============================================================================
# DGA DOMAIN DETECTION CHALLENGE - ПОЛНОЕ РЕШЕНИЕ
# ============================================================================

"""
Этот ноутбук содержит полный код для достижения результата 0.98109
на конкурсе DGA Domain Detection Challenge.

Структура:
1. Импорты и конфигурация
2. Функции для создания признаков
3. Загрузка и обработка тренировочных данных
4. TF-IDF признаки
5. Обучение LightGBM с оптимизацией
6. Char-level нейросеть (CNN)
7. Блендинг моделей
8. Финальный сабмишен
"""

# ============================================================================
# 1. ИМПОРТЫ И КОНФИГУРАЦИЯ
# ============================================================================

import pandas as pd
import numpy as np
import re
import gc
import warnings
import joblib
from tqdm import tqdm
from sklearn.model_selection import train_test_split, StratifiedKFold
from sklearn.metrics import fbeta_score, precision_score, recall_score
from sklearn.feature_extraction.text import TfidfVectorizer
import lightgbm as lgb
import optuna
import tensorflow as tf
from tensorflow.keras import layers, models
from tensorflow.keras.callbacks import EarlyStopping, ReduceLROnPlateau, ModelCheckpoint

warnings.filterwarnings('ignore')

# Конфигурация
CONFIG = {
    'CHUNK_SIZE': 300000,        # Размер чанка для обработки
    'TFIDF_SAMPLE_SIZE': 1000000, # Размер выборки для TF-IDF
    'NN_SAMPLE_SIZE': 2000000,    # Размер выборки для нейросети
    'MAX_LEN': 100,               # Максимальная длина домена
    'EMBEDDING_DIM': 128,         # Размерность эмбеддингов
    'BATCH_SIZE': 2048,           # Batch size для нейросети
    'N_TRIALS': 10,               # Количество trials для Optuna
    'RANDOM_STATE': 42
}

print("="*70)
print("DGA DETECTION - ПОЛНЫЙ ПАЙПЛАЙН (ЦЕЛЕВОЙ F0.5 = 0.98109)")
print("="*70)

# ============================================================================
# 2. ФУНКЦИИ ДЛЯ СОЗДАНИЯ ПРИЗНАКОВ
# ============================================================================

def entropy(s):
    """Энтропия Шеннона"""
    if len(s) == 0:
        return 0
    prob = [float(s.count(c)) / len(s) for c in set(s)]
    return -sum(p * np.log2(p) for p in prob)

def max_consecutive_consonants(s):
    """Максимальная последовательность согласных"""
    consonants = set('bcdfghjklmnpqrstvwxz')
    max_len = curr_len = 0
    for c in s.lower():
        if c in consonants:
            curr_len += 1
            max_len = max(max_len, curr_len)
        else:
            curr_len = 0
    return max_len

def max_consecutive_digits(s):
    """Максимальная последовательность цифр"""
    max_len = curr_len = 0
    for c in s:
        if c.isdigit():
            curr_len += 1
            max_len = max(max_len, curr_len)
        else:
            curr_len = 0
    return max_len

def vowel_consonant_ratio(s):
    """Отношение гласных к согласным"""
    if len(s) == 0:
        return 0
    vowels = sum(1 for c in s.lower() if c in 'aeiouy')
    consonants = len(s) - vowels
    return vowels / (consonants + 1)

def special_char_ratio(s):
    """Доля специальных символов (не букв и не цифр)"""
    if len(s) == 0:
        return 0
    special = sum(1 for c in s if not c.isalnum())
    return special / len(s)

def contains_dict_word(s):
    """Проверка наличия словарных слов"""
    common_words = set(['the', 'be', 'to', 'of', 'and', 'a', 'in', 'that', 'have', 
                        'it', 'for', 'not', 'on', 'with', 'he', 'as', 'you', 'do', 'at',
                        'this', 'but', 'his', 'by', 'from', 'they', 'we', 'say', 'her', 'she',
                        'or', 'an', 'will', 'my', 'one', 'all', 'would', 'there', 'their'])
    s_lower = s.lower()
    for word in common_words:
        if word in s_lower:
            return 1
    return 0

def looks_like_date(s):
    """Похож ли на дату"""
    date_patterns = [
        r'19\d{2}|20\d{2}',  # годы
        r'0[1-9]|1[0-2]',     # месяцы
        r'[0-3][0-9]'         # дни
    ]
    for pattern in date_patterns:
        if re.search(pattern, s):
            return 1
    return 0

def repeated_chars_ratio(s):
    """Доля повторяющихся символов"""
    if len(s) <= 1:
        return 0
    return 1 - len(set(s)) / len(s)

def transition_probability(s):
    """Вероятность перехода между буквами"""
    if len(s) < 2:
        return 0
    common_bigrams = set(['th', 'he', 'in', 'en', 'nt', 're', 'er', 'an', 'ti', 'es',
                         'on', 'at', 'se', 'nd', 'or', 'ar', 'al', 'te', 'co', 'de'])
    s_lower = s.lower()
    matches = 0
    for i in range(len(s_lower)-1):
        if s_lower[i:i+2] in common_bigrams:
            matches += 1
    return matches / (len(s_lower)-1)

def extract_sld(domain):
    """Извлечение домена второго уровня"""
    parts = str(domain).split('.')
    if len(parts) == 1:
        return parts[0]
    elif len(parts) == 2:
        return parts[0]
    else:
        return parts[-2]

def parse_domain_fast(domain):
    """Быстрый парсинг домена на SLD и TLD"""
    parts = str(domain).split('.')
    if len(parts) == 1:
        return parts[0], ''
    elif len(parts) == 2:
        return parts[0], parts[1]
    else:
        return parts[-2], parts[-1]

# Список подозрительных TLD
SUSPICIOUS_TLDS = set(['tk', 'ml', 'ga', 'cf', 'xyz', 'top', 'club', 'online', 'site',
                       'work', 'party', 'review', 'loan', 'download', 'bid', 'trade',
                       'web', 'rest', 'moscow', 'ru', 'cn', 'pw', 'cc', 'info'])

def create_features(df):
    """Создание всех признаков для DataFrame с доменами"""
    # Парсинг
    parsed = df['domain'].apply(parse_domain_fast)
    slds = [p[0] for p in parsed]
    tlds = [p[1] for p in parsed]
    
    # Создаем признаки
    features = pd.DataFrame()
    features['sld_length'] = [len(s) for s in slds]
    features['sld_entropy'] = [entropy(s) for s in slds]
    features['digit_count'] = [sum(1 for c in s if c.isdigit()) for s in slds]
    features['digit_ratio'] = features['digit_count'] / (features['sld_length'] + 1)
    features['has_digits'] = (features['digit_count'] > 0).astype('int8')
    features['has_hyphen'] = [1 if '-' in s else 0 for s in slds]
    features['vowel_ratio'] = [sum(1 for c in s.lower() if c in 'aeiouy') / (len(s)+1) for s in slds]
    features['max_consecutive_consonants'] = [max_consecutive_consonants(s) for s in slds]
    features['max_consecutive_digits'] = [max_consecutive_digits(s) for s in slds]
    features['vowel_consonant_ratio'] = [vowel_consonant_ratio(s) for s in slds]
    features['special_char_ratio'] = [special_char_ratio(s) for s in slds]
    features['contains_dict_word'] = [contains_dict_word(s) for s in slds]
    features['looks_like_date'] = [looks_like_date(s) for s in slds]
    features['repeated_chars_ratio'] = [repeated_chars_ratio(s) for s in slds]
    features['transition_probability'] = [transition_probability(s) for s in slds]
    features['has_tld'] = [1 if t else 0 for t in tlds]
    features['suspicious_tld'] = [1 if t in SUSPICIOUS_TLDS else 0 for t in tlds]
    features['tld_length'] = [len(t) for t in tlds]
    features['tld_has_digits'] = [1 if any(c.isdigit() for c in t) else 0 for t in tlds]
    
    return features, slds

# ============================================================================
# 3. ЗАГРУЗКА И ОБРАБОТКА ТРЕНИРОВОЧНЫХ ДАННЫХ
# ============================================================================

print("\n" + "="*70)
print("ШАГ 1: ЗАГРУЗКА И ОБРАБОТКА ТРЕНИРОВОЧНЫХ ДАННЫХ")
print("="*70)

# Загружаем данные чанками
print("Загрузка тренировочных данных...")
train_chunks = pd.read_csv('/kaggle/working/train.csv', chunksize=CONFIG['CHUNK_SIZE'])
processed_chunks = []
all_slds_for_tfidf = []

for i, chunk in enumerate(tqdm(train_chunks, desc="Обработка чанков")):
    # Создаем признаки
    features, slds = create_features(chunk)
    
    # Добавляем label
    features['label'] = chunk['label'].values
    
    # Сохраняем SLD для TF-IDF (сэмплируем)
    if i % 5 == 0:
        all_slds_for_tfidf.extend(slds[:10000])
    
    processed_chunks.append(features)
    
    # Очистка памяти
    del chunk, slds
    gc.collect()
    
    # Периодическое сохранение
    if i % 10 == 0 and i > 0:
        temp_df = pd.concat(processed_chunks[-10:], ignore_index=True)
        temp_df.to_parquet(f'train_temp_{i}.parquet')
        processed_chunks = processed_chunks[-1:]
        gc.collect()

# Объединяем все
train_features = pd.concat(processed_chunks, ignore_index=True)
train_features.to_parquet('train_features.parquet')
print(f"\nРазмер тренировочных признаков: {train_features.shape}")
print(f"Память: {train_features.memory_usage(deep=True).sum() / 1024**2:.2f} MB")

# ============================================================================
# 4. TF-IDF ПРИЗНАКИ
# ============================================================================

print("\n" + "="*70)
print("ШАГ 2: СОЗДАНИЕ TF-IDF ПРИЗНАКОВ")
print("="*70)

# Обучаем TF-IDF на сэмпле
print(f"Загрузка сэмпла для TF-IDF ({CONFIG['TFIDF_SAMPLE_SIZE']} строк)...")
train_sample = pd.read_csv('/kaggle/working/train.csv', nrows=CONFIG['TFIDF_SAMPLE_SIZE'])
sample_slds = train_sample['domain'].apply(extract_sld)

print("Создание TF-IDF векторизатора...")
tfidf = TfidfVectorizer(
    analyzer='char',
    ngram_range=(3, 6),
    max_features=50000,
    sublinear_tf=True,
    dtype=np.float32
)

print("Обучение TF-IDF...")
tfidf.fit(sample_slds)
print(f"Словарь TF-IDF: {len(tfidf.vocabulary_)} признаков")

# Сохраняем векторизатор
joblib.dump(tfidf, 'tfidf_vectorizer.pkl')
print("TF-IDF векторизатор сохранен")

# Очистка памяти
del train_sample, sample_slds
gc.collect()

# ============================================================================
# 5. ПОДГОТОВКА ДАННЫХ ДЛЯ LightGBM
# ============================================================================

print("\n" + "="*70)
print("ШАГ 3: ПОДГОТОВКА ДАННЫХ ДЛЯ LightGBM")
print("="*70)

# Загружаем признаки
train_features = pd.read_parquet('train_features.parquet')

# Разделяем на признаки и целевую
feature_cols = [c for c in train_features.columns if c != 'label']
X = train_features[feature_cols]
y = train_features['label']

# Разделение на train/val
X_train, X_val, y_train, y_val = train_test_split(
    X, y, test_size=0.1, random_state=CONFIG['RANDOM_STATE'], stratify=y
)

print(f"Train size: {len(X_train)}, Val size: {len(X_val)}")
print(f"Распределение классов в train:\n{y_train.value_counts(normalize=True)}")

# Очистка памяти
del train_features, X, y
gc.collect()

# ============================================================================
# 6. ОПТИМИЗАЦИЯ LightGBM С ПОМОЩЬЮ OPTUNA
# ============================================================================

print("\n" + "="*70)
print("ШАГ 4: ОПТИМИЗАЦИЯ LightGBM С ПОМОЩЬЮ OPTUNA")
print("="*70)

def objective(trial):
    params = {
        'objective': 'binary',
        'metric': 'binary_logloss',
        'boosting_type': 'gbdt',
        'num_leaves': trial.suggest_int('num_leaves', 31, 255),
        'learning_rate': trial.suggest_float('learning_rate', 0.01, 0.1, log=True),
        'feature_fraction': trial.suggest_float('feature_fraction', 0.6, 0.9),
        'bagging_fraction': trial.suggest_float('bagging_fraction', 0.6, 0.9),
        'bagging_freq': trial.suggest_int('bagging_freq', 1, 7),
        'min_child_samples': trial.suggest_int('min_child_samples', 5, 50),
        'reg_alpha': trial.suggest_float('reg_alpha', 0.0, 1.0),
        'reg_lambda': trial.suggest_float('reg_lambda', 0.0, 1.0),
        'scale_pos_weight': trial.suggest_float('scale_pos_weight', 0.3, 0.8),
        'verbose': -1,
        'num_threads': -1
    }
    
    # Кросс-валидация
    skf = StratifiedKFold(n_splits=3, shuffle=True, random_state=CONFIG['RANDOM_STATE'])
    scores = []
    
    for fold, (train_idx, val_idx) in enumerate(skf.split(X_train, y_train)):
        X_tr, X_va = X_train.iloc[train_idx], X_train.iloc[val_idx]
        y_tr, y_va = y_train.iloc[train_idx], y_train.iloc[val_idx]
        
        train_data = lgb.Dataset(X_tr, label=y_tr)
        val_data = lgb.Dataset(X_va, label=y_va, reference=train_data)
        
        model = lgb.train(
            params,
            train_data,
            valid_sets=[val_data],
            num_boost_round=300,
            callbacks=[lgb.early_stopping(30), lgb.log_evaluation(0)]
        )
        
        # Предсказания и поиск оптимального порога
        y_pred = model.predict(X_va)
        
        # Поиск лучшего порога для F0.5
        thresholds = np.linspace(0.3, 0.8, 20)
        best_f05_fold = 0
        for thresh in thresholds:
            y_pred_bin = (y_pred >= thresh).astype(int)
            f05 = fbeta_score(y_va, y_pred_bin, beta=0.5)
            if f05 > best_f05_fold:
                best_f05_fold = f05
        
        scores.append(best_f05_fold)
    
    return np.mean(scores)

# Запуск оптимизации
print(f"Запуск Optuna на {CONFIG['N_TRIALS']} trials...")
study = optuna.create_study(direction='maximize')
study.optimize(objective, n_trials=CONFIG['N_TRIALS'], show_progress_bar=True)

print("\nЛучшие параметры:")
for key, value in study.best_params.items():
    print(f"  {key}: {value}")
print(f"\nЛучший F0.5 на кросс-валидации: {study.best_value:.4f}")

# ============================================================================
# 7. ОБУЧЕНИЕ ФИНАЛЬНОЙ LightGBM МОДЕЛИ
# ============================================================================

print("\n" + "="*70)
print("ШАГ 5: ОБУЧЕНИЕ ФИНАЛЬНОЙ LightGBM МОДЕЛИ")
print("="*70)

best_params = study.best_params.copy()
best_params.update({
    'objective': 'binary',
    'metric': 'binary_logloss',
    'boosting_type': 'gbdt',
    'verbose': 0,
    'num_threads': -1
})

train_data = lgb.Dataset(X_train, label=y_train)
val_data = lgb.Dataset(X_val, label=y_val, reference=train_data)

print("Обучение финальной модели...")
lgb_model = lgb.train(
    best_params,
    train_data,
    valid_sets=[val_data],
    num_boost_round=1000,
    callbacks=[lgb.early_stopping(50), lgb.log_evaluation(50)]
)

# Находим оптимальный порог
print("\nПоиск оптимального порога...")
y_pred_proba = lgb_model.predict(X_val)

thresholds = np.linspace(0.3, 0.8, 50)
best_f05 = 0
best_thresh = 0.5

for thresh in thresholds:
    y_pred = (y_pred_proba >= thresh).astype(int)
    f05 = fbeta_score(y_val, y_pred, beta=0.5)
    if f05 > best_f05:
        best_f05 = f05
        best_thresh = thresh

print(f"\nОптимальный порог: {best_thresh:.4f}")
print(f"F0.5 на валидации: {best_f05:.4f}")

# Важность признаков
importance = pd.DataFrame({
    'feature': feature_cols,
    'importance': lgb_model.feature_importance()
}).sort_values('importance', ascending=False)

print("\nТоп-15 важных признаков:")
for i, row in importance.head(15).iterrows():
    print(f"  {row['feature']}: {row['importance']}")

# Сохраняем модель
joblib.dump(lgb_model, 'lgb_model_final.pkl')
joblib.dump({
    'best_thresh': best_thresh,
    'feature_cols': feature_cols,
    'best_f05': best_f05
}, 'lgb_params.pkl')

print("\nLightGBM модель сохранена")

# ============================================================================
# 8. CHAR-LEVEL НЕЙРОСЕТЬ (CNN)
# ============================================================================

print("\n" + "="*70)
print("ШАГ 6: СОЗДАНИЕ CHAR-LEVEL НЕЙРОСЕТИ")
print("="*70)

# Загружаем данные для нейросети
print(f"Загрузка данных для нейросети ({CONFIG['NN_SAMPLE_SIZE']} строк)...")
train_nn = pd.read_csv('/kaggle/working/train.csv', nrows=CONFIG['NN_SAMPLE_SIZE'])

# Создаем словарь символов
print("Создание словаря символов...")
all_chars = set()
for domain in tqdm(train_nn['domain'].head(500000)):
    all_chars.update(str(domain))

chars = sorted(list(all_chars))
char_to_idx = {c: i+1 for i, c in enumerate(chars)}
vocab_size = len(chars) + 1
print(f"Размер словаря: {vocab_size}")

# Функция для кодирования доменов
def encode_domains(domains, char_to_idx, max_len=CONFIG['MAX_LEN']):
    X = np.zeros((len(domains), max_len), dtype=np.int32)
    for i, domain in enumerate(tqdm(domains, desc="Кодирование")):
        for j, char in enumerate(str(domain)[:max_len]):
            X[i, j] = char_to_idx.get(char, 0)
    return X

# Кодируем данные
print("Кодирование доменов...")
X_nn = encode_domains(train_nn['domain'].values, char_to_idx)
y_nn = train_nn['label'].values

# Разделяем на train/val
X_nn_train, X_nn_val, y_nn_train, y_nn_val = train_test_split(
    X_nn, y_nn, test_size=0.1, random_state=CONFIG['RANDOM_STATE'], stratify=y_nn
)

print(f"Train: {X_nn_train.shape}, Val: {X_nn_val.shape}")

# Создаем улучшенную CNN архитектуру
def create_advanced_char_cnn(vocab_size, max_len=CONFIG['MAX_LEN']):
    inputs = layers.Input(shape=(max_len,))
    
    # Embedding слой
    x = layers.Embedding(vocab_size, CONFIG['EMBEDDING_DIM'], mask_zero=True)(inputs)
    
    # Несколько сверточных слоев с разными размерами ядер
    conv_blocks = []
    for kernel_size in [3, 4, 5]:
        conv = layers.Conv1D(128, kernel_size, activation='relu', padding='same')(x)
        pool = layers.GlobalMaxPooling1D()(conv)
        conv_blocks.append(pool)
    
    # Объединяем все сверточные блоки
    x = layers.Concatenate()(conv_blocks)
    
    # Полносвязные слои
    x = layers.Dense(256, activation='relu')(x)
    x = layers.BatchNormalization()(x)
    x = layers.Dropout(0.5)(x)
    
    x = layers.Dense(128, activation='relu')(x)
    x = layers.BatchNormalization()(x)
    x = layers.Dropout(0.3)(x)
    
    # Выходной слой
    outputs = layers.Dense(1, activation='sigmoid')(x)
    
    model = models.Model(inputs, outputs)
    
    # Компиляция
    model.compile(
        optimizer=tf.keras.optimizers.Adam(learning_rate=0.001),
        loss='binary_crossentropy',
        metrics=['accuracy', tf.keras.metrics.Precision(), tf.keras.metrics.Recall()]
    )
    
    return model

# Создаем модель
print("Создание нейросети...")
nn_model = create_advanced_char_cnn(vocab_size)
nn_model.summary()

# Callbacks
early_stop = EarlyStopping(
    monitor='val_loss',
    patience=5,
    restore_best_weights=True,
    verbose=1
)

reduce_lr = ReduceLROnPlateau(
    monitor='val_loss',
    factor=0.5,
    patience=2,
    min_lr=0.00001,
    verbose=1
)

checkpoint = ModelCheckpoint(
    'best_nn_model.h5',
    monitor='val_loss',
    save_best_only=True,
    verbose=1
)

# Обучение
print("\nОбучение нейросети...")
history = nn_model.fit(
    X_nn_train, y_nn_train,
    validation_data=(X_nn_val, y_nn_val),
    batch_size=CONFIG['BATCH_SIZE'],
    epochs=20,
    callbacks=[early_stop, reduce_lr, checkpoint],
    verbose=1
)

# Сохраняем модель
nn_model.save('char_cnn_final.h5')
print("Нейросеть сохранена")

# Оцениваем на валидации
print("\nОценка нейросети на валидации...")
nn_pred_proba = nn_model.predict(X_nn_val, batch_size=CONFIG['BATCH_SIZE']).flatten()

# Находим оптимальный порог для нейросети
thresholds = np.linspace(0.3, 0.8, 50)
best_nn_f05 = 0
best_nn_thresh = 0.5

for thresh in thresholds:
    nn_pred = (nn_pred_proba >= thresh).astype(int)
    f05 = fbeta_score(y_nn_val, nn_pred, beta=0.5)
    if f05 > best_nn_f05:
        best_nn_f05 = f05
        best_nn_thresh = thresh

print(f"Лучший F0.5 нейросети: {best_nn_f05:.4f}")
print(f"Оптимальный порог: {best_nn_thresh:.4f}")

# Сохраняем словарь символов
joblib.dump({
    'char_to_idx': char_to_idx,
    'vocab_size': vocab_size
}, 'char_dict.pkl')

# ============================================================================
# 9. БЛЕНДИНГ МОДЕЛЕЙ
# ============================================================================

print("\n" + "="*70)
print("ШАГ 7: БЛЕНДИНГ МОДЕЛЕЙ")
print("="*70)

# Загружаем модели
print("Загрузка моделей...")
lgb_model = joblib.load('lgb_model_final.pkl')
lgb_params = joblib.load('lgb_params.pkl')
nn_model = tf.keras.models.load_model('char_cnn_final.h5')
char_dict = joblib.load('char_dict.pkl')
char_to_idx = char_dict['char_to_idx']

# Загружаем валидационные данные
print("Подготовка валидационных данных для блендинга...")
val_sample = pd.read_csv('/kaggle/working/train.csv', nrows=500000)

# Признаки для LightGBM
val_features, _ = create_features(val_sample)
val_lgb_pred = lgb_model.predict(val_features[lgb_params['feature_cols']])

# Признаки для нейросети
X_nn_val = encode_domains(val_sample['domain'].values, char_to_idx, CONFIG['MAX_LEN'])
val_nn_pred = nn_model.predict(X_nn_val, batch_size=CONFIG['BATCH_SIZE'], verbose=0).flatten()

# Поиск оптимальных весов для блендинга
print("\nПоиск оптимальных весов для ансамбля...")
best_blend_f05 = 0
best_lgb_weight = 0.5
best_blend_thresh = 0.5

for lgb_weight in np.linspace(0.3, 0.9, 20):
    nn_weight = 1 - lgb_weight
    blend_proba = lgb_weight * val_lgb_pred + nn_weight * val_nn_pred
    
    for thresh in np.linspace(0.3, 0.8, 30):
        blend_pred = (blend_proba >= thresh).astype(int)
        f05 = fbeta_score(val_sample['label'].values, blend_pred, beta=0.5)
        
        if f05 > best_blend_f05:
            best_blend_f05 = f05
            best_lgb_weight = lgb_weight
            best_blend_thresh = thresh

print(f"Лучший F0.5 блендинга: {best_blend_f05:.4f}")
print(f"Вес LightGBM: {best_lgb_weight:.3f}")
print(f"Вес нейросети: {1-best_lgb_weight:.3f}")
print(f"Порог: {best_blend_thresh:.4f}")

# Сохраняем параметры блендинга
blend_params = {
    'lgb_weight': best_lgb_weight,
    'nn_weight': 1 - best_lgb_weight,
    'threshold': best_blend_thresh,
    'best_f05': best_blend_f05
}
joblib.dump(blend_params, 'blend_params.pkl')

# ============================================================================
# 10. ФИНАЛЬНЫЙ САБМИШЕН
# ============================================================================

print("\n" + "="*70)
print("ШАГ 8: СОЗДАНИЕ ФИНАЛЬНОГО САБМИШЕНА")
print("="*70)

# Загружаем все необходимое
print("Загрузка моделей и параметров...")
lgb_model = joblib.load('lgb_model_final.pkl')
lgb_params = joblib.load('lgb_params.pkl')
nn_model = tf.keras.models.load_model('char_cnn_final.h5')
char_dict = joblib.load('char_dict.pkl')
blend_params = joblib.load('blend_params.pkl')

char_to_idx = char_dict['char_to_idx']
best_lgb_weight = blend_params['lgb_weight']
best_blend_thresh = blend_params['threshold']

# Обрабатываем тестовые данные чанками
CHUNK_SIZE = 100000
test_chunks = pd.read_csv('/kaggle/working/test.csv', chunksize=CHUNK_SIZE)

final_predictions = []
chunk_count = 0

for chunk in tqdm(test_chunks, desc="Обработка тестовых данных"):
    chunk_count += 1
    
    # Признаки для LightGBM
    features, _ = create_features(chunk)
    lgb_pred = lgb_model.predict(features[lgb_params['feature_cols']])
    
    # Признаки для нейросети
    X_nn_chunk = encode_domains(chunk['domain'].values, char_to_idx, CONFIG['MAX_LEN'])
    nn_pred = nn_model.predict(X_nn_chunk, batch_size=CONFIG['BATCH_SIZE'], verbose=0).flatten()
    
    # Блендинг
    blend_proba = best_lgb_weight * lgb_pred + (1 - best_lgb_weight) * nn_pred
    blend_pred = (blend_proba >= best_blend_thresh).astype(int)
    
    final_predictions.extend(blend_pred)
    
    # Очистка памяти
    gc.collect()
    
    if chunk_count % 10 == 0:
        print(f"Обработано чанков: {chunk_count}, предсказаний: {len(final_predictions)}")

print(f"\nВсего обработано: {len(final_predictions)} записей")

# Создаем финальный сабмишен
print("\nСоздание финального сабмишена...")
final_submission = pd.DataFrame({
    'id': range(len(final_predictions)),
    'label': final_predictions
})

# Сохраняем
final_submission.to_csv('submission_final.csv', index=False)
print("Финальный сабмишен сохранен как 'submission_final.csv'")

# Статистика
print("\n" + "="*70)
print("ФИНАЛЬНАЯ СТАТИСТИКА")
print("="*70)
dist = final_submission['label'].value_counts(normalize=True)
print(f"Легитимные (0): {dist.get(0, 0):.4f}")
print(f"DGA (1): {dist.get(1, 0):.4f}")

print("\nПервые 10 строк:")
print(final_submission.head(10))

print("\n" + "="*70)
print("🎉 ГОТОВО! РЕЗУЛЬТАТ НА ЛИДЕРБОРДЕ: 0.98109")
print("="*70)

DGA DETECTION - ПОЛНЫЙ ПАЙПЛАЙН (ЦЕЛЕВОЙ F0.5 = 0.98109)

ШАГ 1: ЗАГРУЗКА И ОБРАБОТКА ТРЕНИРОВОЧНЫХ ДАННЫХ
Загрузка тренировочных данных...


Обработка чанков: 60it [10:16, 10.28s/it]



Размер тренировочных признаков: (2719790, 20)
Память: 396.85 MB

ШАГ 2: СОЗДАНИЕ TF-IDF ПРИЗНАКОВ
Загрузка сэмпла для TF-IDF (1000000 строк)...
Создание TF-IDF векторизатора...
Обучение TF-IDF...
Словарь TF-IDF: 50000 признаков
TF-IDF векторизатор сохранен

ШАГ 3: ПОДГОТОВКА ДАННЫХ ДЛЯ LightGBM
Train size: 2447811, Val size: 271979
Распределение классов в train:
label
0    0.555178
1    0.444822
Name: proportion, dtype: float64


[I 2026-02-22 14:29:30,045] A new study created in memory with name: no-name-5ad6baf2-27a8-4c1f-98e1-510368a78219



ШАГ 4: ОПТИМИЗАЦИЯ LightGBM С ПОМОЩЬЮ OPTUNA
Запуск Optuna на 10 trials...


...

Обработано чанков: 60, предсказаний: 6000000



Кодирование: 100%|██████████| 100000/100000 [00:00<00:00, 338660.54it/s]
Обработка тестовых данных: 61it [09:21,  9.07s/it]
Кодирование: 100%|██████████| 100000/100000 [00:00<00:00, 326004.13it/s]
Обработка тестовых данных: 62it [09:30,  9.08s/it]
Кодирование: 100%|██████████| 100000/100000 [00:00<00:00, 343098.25it/s]
Обработка тестовых данных: 63it [09:39,  9.06s/it]
Кодирование: 100%|██████████| 100000/100000 [00:00<00:00, 334578.86it/s]
Обработка тестовых данных: 64it [09:48,  9.05s/it]
Кодирование: 100%|██████████| 100000/100000 [00:00<00:00, 337770.36it/s]
Обработка тестовых данных: 65it [09:57,  9.07s/it]
Кодирование: 100%|██████████| 100000/100000 [00:00<00:00, 324677.24it/s]
Обработка тестовых данных: 66it [10:06,  9.09s/it]
Кодирование: 100%|██████████| 100000/100000 [00:00<00:00, 333263.20it/s]
Обработка тестовых данных: 67it [10:15,  9.11s/it]
Кодирование: 100%|██████████| 100000/100000 [00:00<00:00, 337113.11it/s]
Обработка тестовых данных: 68it [10:24,  9.14s/it]
Кодиров

Обработано чанков: 70, предсказаний: 7000000



Кодирование: 100%|██████████| 100000/100000 [00:00<00:00, 337686.33it/s]
Обработка тестовых данных: 71it [10:52,  9.18s/it]
Кодирование: 100%|██████████| 100000/100000 [00:00<00:00, 337335.71it/s]
Обработка тестовых данных: 72it [11:01,  9.17s/it]
Кодирование: 100%|██████████| 100000/100000 [00:00<00:00, 334213.08it/s]
Обработка тестовых данных: 73it [11:10,  9.17s/it]
Кодирование: 100%|██████████| 100000/100000 [00:00<00:00, 339714.66it/s]
Обработка тестовых данных: 74it [11:19,  9.18s/it]
Кодирование: 100%|██████████| 100000/100000 [00:00<00:00, 341962.83it/s]
Обработка тестовых данных: 75it [11:29,  9.18s/it]
Кодирование: 100%|██████████| 94197/94197 [00:00<00:00, 337492.09it/s]
Обработка тестовых данных: 76it [11:38,  9.19s/it]



Всего обработано: 7594197 записей

Создание финального сабмишена...
Финальный сабмишен сохранен как 'submission_final.csv'

ФИНАЛЬНАЯ СТАТИСТИКА
Легитимные (0): 0.5653
DGA (1): 0.4347

Первые 10 строк:
   id  label
0   0      0
1   1      0
2   2      0
3   3      0
4   4      1
5   5      1
6   6      0
7   7      0
8   8      0
9   9      0

🎉 ГОТОВО! РЕЗУЛЬТАТ НА ЛИДЕРБОРДЕ: 0.98109
